# Aula 1 — Regressão Linear Simples

Como usar este notebook: a **Parte A** tem células que o professor roda e
explica durante a aula. Acompanhe na tela, sem precisar digitar nada. A
**Parte B** é com você: complete os exercícios nos lugares marcados com
`# SEU CODIGO AQUI`.

Se ainda não sabe como abrir e salvar sua própria cópia deste notebook,
veja a página **Antes de começar** no material da aula antes de continuar.

## Parte A: Demonstração

### Os dados: 200 corridas de aplicativo

Cada linha é uma corrida, com a distância em km e o preço em reais.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Endereço dos dados desta aula no GitHub.
URL_DADOS = "https://raw.githubusercontent.com/klein-natan/nanodegree-AI-Atitus/main/data/corridas_app.csv"
# Alternativa para testar offline, antes do repositório existir no GitHub
# (ou durante a aula, se a internet falhar):
# URL_DADOS = "../../data/corridas_app.csv"

dados = pd.read_csv(URL_DADOS)
dados.head()

In [ ]:
dados.describe()

In [ ]:
plt.scatter(dados["distancia_km"], dados["preco"])
plt.xlabel("Distância (km)")
plt.ylabel("Preço (R$)")
plt.title("Distância x preço de cada corrida")
plt.show()

### A reta de preço e o seu erro

A reta que queremos ajustar é:

$$\text{preço} = w_0 + w_1 \cdot \text{distância}$$

O resíduo de uma corrida é o preço real menos o preço previsto. Ele se
escreve com a letra grega épsilon:

$$\varepsilon_i = y_i - \hat{y}_i$$

Aqui $y_i$ é o preço real da corrida $i$ e $\hat{y}_i$ é o preço que a
reta previu para ela.

O erro dela sobre as 200 corridas é o erro quadrático médio, a média dos
resíduos elevados ao quadrado:

$$\text{MSE} = \frac{1}{n}\sum_{i=1}^{n}\left(y_i - \hat{y}_i\right)^2$$

In [ ]:
def calcular_mse(w0, w1, distancias, precos):
    # Calcula o preço que a reta prevê para cada corrida
    precos_previstos = w0 + w1 * distancias
    # Calcula o erro de cada corrida e eleva ao quadrado
    erro_quadratico = (precos - precos_previstos) ** 2
    # Tira a média dos erros ao quadrado
    return erro_quadratico.mean()

distancias = dados["distancia_km"]
precos = dados["preco"]

chutes = [
    ("Baixa demais", 0.0, 1.0),
    ("Boa aproximação", 4.6, 2.2),
    ("Alta demais", 10.0, 3.5),
]

for nome, w0, w1 in chutes:
    mse = calcular_mse(w0, w1, distancias, precos)
    print(f"{nome}: w0={w0}, w1={w1} -> MSE = {mse:.2f}")

### O terreno do erro

Congelando $w_0$ e variando só $w_1$, o MSE desenha uma curva com um
fundo. O melhor $w_1$ é o ponto mais baixo dessa curva.

In [ ]:
import numpy as np

# Congela w0 no valor certo e testa vários w1: é a curva do erro
w0_fixo = 4.63
valores_w1 = np.arange(0.3, 4.4, 0.05)
erros = []
for w1_teste in valores_w1:
    erros.append(calcular_mse(w0_fixo, w1_teste, distancias, precos))

plt.plot(valores_w1, erros)
plt.xlabel("w1 (preço por km)")
plt.ylabel("Erro (MSE)")
plt.title("O erro em função de um único peso")
plt.show()

### A inclinação (a derivada) em um ponto

Você está parado num ponto da curva e precisa decidir uma coisa só: para
que lado andar. A derivada responde exatamente isso. Ela diz o que
acontece com o erro se você aumentar o peso um tiquinho.

A conta é a subida dividida pelo avanço, com um avanço $h$ bem pequeno:

$$\text{inclinação} \approx \frac{\text{MSE}(w_1 + h) - \text{MSE}(w_1)}{h}$$

O número que sai carrega duas informações:

- **O sinal** diz o lado. Positivo: aumentar o peso aumenta o erro, então
  o modelo anda para o outro lado. Negativo: aumentar o peso reduz o erro.
- **O tamanho** diz a pressa. Grande é ladeira íngreme. Perto de zero é
  chão plano, e o fundo está perto.

Sem esse número, você teria que testar valores no escuro, um por um.

In [ ]:
# A inclinação em um ponto: ande um tiquinho para a frente e veja o quanto o
# erro subiu. Subida dividida por avanço é a inclinação (a derivada).
w1_ponto = 3.2
tiquinho = 0.05
erro_aqui = calcular_mse(w0_fixo, w1_ponto, distancias, precos)
erro_adiante = calcular_mse(w0_fixo, w1_ponto + tiquinho, distancias, precos)
inclinacao = (erro_adiante - erro_aqui) / tiquinho

print(f"Erro com w1 = {w1_ponto}: {erro_aqui:.2f}")
print(f"Erro com w1 = {w1_ponto + tiquinho}: {erro_adiante:.2f}")
print(f"Inclinação nesse ponto: {inclinacao:.1f}")
print("Inclinação positiva: para reduzir o erro, w1 precisa diminuir.")

In [ ]:
# Desenha a mesma curva com a reta tangente no ponto: a inclinação da
# tangente é o número que o gradiente descendente usa a cada passo
reta_tangente = erro_aqui + inclinacao * (valores_w1 - w1_ponto)

plt.plot(valores_w1, erros)
plt.plot(valores_w1, reta_tangente)
plt.scatter([w1_ponto], [erro_aqui])
plt.ylim(0, 300)
plt.xlabel("w1 (preço por km)")
plt.ylabel("Erro (MSE)")
plt.title("A reta tangente em w1 = 3,2")
plt.show()

### Gradiente descendente

Com dois pesos existe uma inclinação para cada um. Essa lista de
inclinações é o gradiente:

$$\nabla \text{MSE} = \left( \frac{\partial \text{MSE}}{\partial w_0} \; , \; \frac{\partial \text{MSE}}{\partial w_1} \right)$$

E a regra de atualização anda contra ele, um passo de cada vez:

$$w_{t+1} = w_t - \alpha \cdot \frac{\partial \text{MSE}}{\partial w}$$

O $t$ é o número do passo e $\alpha$ é a taxa de aprendizado, o tamanho
do passo.

O algoritmo inteiro cabe em cinco passos, e é exatamente isso que o laço
da próxima célula faz:

1. Chute qualquer `w0` e `w1`. Zero serve.
2. Calcule o erro (o MSE) com os pesos de agora.
3. Calcule a inclinação do erro para cada peso.
4. Ande contra a inclinação, uma vez para cada peso.
5. Volte ao passo 2 e repita.

Por que não usar sempre a fórmula fechada dos mínimos quadrados, que dá a
resposta de primeira? Porque ela só existe para modelos pequenos. Uma rede
neural tem milhões de pesos, e ninguém resolveu essa conta no papel. O
gradiente descendente não precisa resolver conta nenhuma: ele só precisa
saber, de onde está, para que lado o erro diminui.

De onde vêm as duas linhas do gradiente? Da mesma conta de "subida
dividida por avanço" que fizemos acima, feita com álgebra em vez de com
um tiquinho. O resultado é uma fórmula fechada: para o MSE de uma reta,
a inclinação em relação a `w0` é `-2 × média(erro)` e em relação a `w1` é
`-2 × média(erro × distância)`. Você não precisa deduzir isso. Precisa
saber que é o mesmo número que a célula do tiquinho calcula, só que
exato e rápido.

In [ ]:
def um_passo_gradiente(w0, w1, distancias, precos, alpha):
    # Calcula o preço previsto e o erro de cada corrida
    precos_previstos = w0 + w1 * distancias
    erro = precos - precos_previstos
    # A inclinação do erro em relação a cada peso (a fórmula fechada da
    # derivada do MSE: o mesmo número da conta do tiquinho, só que exato)
    gradiente_w0 = -2 * erro.mean()
    gradiente_w1 = -2 * (erro * distancias).mean()
    # Dá um passo na direção que reduz o erro
    novo_w0 = w0 - alpha * gradiente_w0
    novo_w1 = w1 - alpha * gradiente_w1
    return novo_w0, novo_w1

w0, w1 = 0.0, 0.0
alpha = 0.01
historico_perda = []

for passo in range(300):
    mse_passo = calcular_mse(w0, w1, distancias, precos)
    historico_perda.append(mse_passo)
    w0, w1 = um_passo_gradiente(w0, w1, distancias, precos, alpha)

print(f"Depois de 300 passos: w0 = {w0:.2f}, w1 = {w1:.2f}")
print(f"MSE final: {historico_perda[-1]:.2f}")

plt.plot(historico_perda)
plt.xlabel("Passo do gradiente descendente")
plt.ylabel("MSE")
plt.title("A perda caindo a cada passo")
plt.show()

### O atalho: mínimos quadrados

Para uma reta simples existe fórmula fechada, e é ela que o
`scikit-learn` usa por baixo dos panos:

$$w_1 = \frac{\sum_{i=1}^{n}(x_i - \bar{x})(y_i - \bar{y})}{\sum_{i=1}^{n}(x_i - \bar{x})^2} \qquad w_0 = \bar{y} - w_1 \bar{x}$$

In [ ]:
from sklearn.linear_model import LinearRegression

modelo = LinearRegression()
modelo.fit(dados[["distancia_km"]], dados["preco"])

print(f"w0 (taxa fixa) encontrado pelo scikit-learn: {modelo.intercept_:.2f}")
print(f"w1 (preço por km) encontrado pelo scikit-learn: {modelo.coef_[0]:.2f}")

### As métricas, em três unidades diferentes

MAE e RMSE saem em reais. O MAPE sai em porcentagem, e responde outra
pergunta: o modelo erra cerca de R$ 1,60 por corrida, e isso é muito?

Depende da corrida. R$ 1,60 numa corrida de R$ 8 é um quinto do preço, e
o cliente percebe. Os mesmos R$ 1,60 numa corrida de R$ 50 são pouco mais
de 3%, e ninguém reclama.

O MAPE resolve isso com uma ideia só: antes de tirar a média, o erro de
cada corrida vira porcentagem do preço daquela corrida. Numa corrida de
R$ 6 que a reta errou em R$ 0,50, a conta é 0,50 ÷ 6,00 = 8,3%.

$$\text{MAPE} = \frac{100}{n}\sum_{i=1}^{n}\left|\frac{y_i - \hat{y}_i}{y_i}\right|$$

Leia a fórmula de dentro para fora: pegue o resíduo, divida pelo preço
real, tire o sinal, some tudo, divida pelo número de corridas e
multiplique por 100.

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

precos_previstos = modelo.predict(dados[["distancia_km"]])
residuos = dados["preco"] - precos_previstos

mae_modelo = mean_absolute_error(dados["preco"], precos_previstos)
rmse_modelo = mean_squared_error(dados["preco"], precos_previstos) ** 0.5
mape_modelo = 100 * (residuos / dados["preco"]).abs().mean()

print(f"MAE:  R$ {mae_modelo:.2f}")
print(f"RMSE: R$ {rmse_modelo:.2f}")
print(f"MAPE: {mape_modelo:.1f}%")
print(f"R²:   {r2_score(dados['preco'], precos_previstos):.3f}")

In [ ]:
# O MAPE conta uma historia que o MAE esconde: separe por faixa de distancia
faixas = pd.cut(dados["distancia_km"], bins=[0, 5, 10, 15, 30])
resumo = pd.DataFrame({
    "corridas": dados.groupby(faixas, observed=True)["preco"].size(),
    "preco_medio": dados.groupby(faixas, observed=True)["preco"].mean(),
    "MAE": residuos.abs().groupby(faixas, observed=True).mean(),
    "MAPE": 100 * (residuos / dados["preco"]).abs().groupby(faixas, observed=True).mean(),
})
print(resumo.round(2))
print()
print("O MAE quase não muda entre as faixas. O MAPE muda quase quatro vezes.")

### A faixa: o modelo acerta a média, não a sua corrida

Para 8 km existem corridas de R$ 19 e de R$ 26, todas de verdade. A reta
acerta a média delas, e nenhuma corrida sozinha é a média.

O tamanho típico de um resíduo se chama $s$:

$$s = \sqrt{\frac{1}{n-2}\sum_{i=1}^{n}(y_i - \hat{y}_i)^2}$$

E a faixa em que cai uma corrida é $\hat{y} \pm 2s$.

In [ ]:
n = len(dados)
s_residual = ((residuos ** 2).sum() / (n - 2)) ** 0.5
margem = 2 * s_residual

print(f"s (resíduo típico): R$ {s_residual:.2f}")
print(f"margem (2s):        R$ {margem:.2f}")
print()

previsao_8km = modelo.predict(pd.DataFrame({"distancia_km": [8.0]}))[0]
print(f"Corrida de 8 km: R$ {previsao_8km:.2f}")
print(f"Faixa: de R$ {previsao_8km - margem:.2f} a R$ {previsao_8km + margem:.2f}")

In [ ]:
# A faixa funciona? Conte quantas corridas caem dentro dela
dentro = residuos.abs() <= margem
print(f"{dentro.sum()} das {n} corridas caem na faixa ({100 * dentro.mean():.1f}%)")
print("A faixa prometia 95%. Conferir é só contar.")

plt.figure(figsize=(9, 5.5))
grade = np.linspace(0, dados["distancia_km"].max(), 100)
reta_grade = modelo.intercept_ + modelo.coef_[0] * grade
plt.fill_between(grade, reta_grade - margem, reta_grade + margem, alpha=0.2)
plt.scatter(dados["distancia_km"], dados["preco"], s=15)
plt.plot(grade, reta_grade)
plt.xlabel("Distância (km)")
plt.ylabel("Preço (R$)")
plt.title("A reta e a faixa onde caem 95% das corridas")
plt.show()

## Parte B: Exercícios

Complete cada exercício no espaço marcado com `# SEU CODIGO AQUI`. Rode a
célula de verificação logo depois para conferir sua resposta. Ela nunca
mostra a solução, só um sinal de certo ou errado e uma dica.

### Exercício 1: conhecendo os dados

Rode a célula abaixo e observe: quantas corridas existem? Qual é a
distância da corrida mais longa? Qual é o preço mais alto?

In [ ]:
dados.describe()

In [ ]:
if len(dados) == 200:
    print(f"✅ Os dados têm {len(dados)} corridas, como esperado.")
else:
    print("❌ Confira se você rodou a célula que carrega os dados, no início do notebook.")

### Exercício 2: a relação parece linear?

Rode a célula abaixo. Olhando o gráfico, você diria que uma reta
representa bem esses dados?

In [ ]:
plt.scatter(dados["distancia_km"], dados["preco"])
plt.xlabel("Distância (km)")
plt.ylabel("Preço (R$)")
plt.title("Distância x preço")
plt.show()

In [ ]:
print("Converse com um colega ou o professor: os pontos parecem seguir uma reta?")

### Exercício 3: calculando o MSE na mão

Complete a função `meu_mse`, que calcula:

$$\text{MSE} = \frac{1}{n}\sum_{i=1}^{n}\left(y_i - \hat{y}_i\right)^2$$

São três linhas: prever o preço de cada corrida, elevar o erro ao
quadrado e tirar a média. É a mesma conta da demonstração, agora é você
quem escreve.

In [ ]:
w0_teste, w1_teste = 4.6, 2.2

In [ ]:
# SEU CODIGO AQUI

In [ ]:
mse_calculado = meu_mse(w0_teste, w1_teste, dados["distancia_km"], dados["preco"])
print(f"Seu MSE: {mse_calculado:.2f}")

In [ ]:
mse_esperado = calcular_mse(4.6, 2.2, dados["distancia_km"], dados["preco"])
if abs(mse_calculado - mse_esperado) < 0.01:
    print(f"✅ Seu MSE ({mse_calculado:.2f}) bate com o esperado.")
else:
    print("❌ Confira: você elevou o erro ao quadrado antes de tirar a média?")

### Exercício 4: testando taxas de aprendizado

Cada passo do gradiente descendente aplica:

$$w_{t+1} = w_t - \alpha \cdot \frac{\partial \text{MSE}}{\partial w}$$

Complete o laço que roda 100 passos, usando a função
`um_passo_gradiente` já pronta (vista na demonstração). Depois, troque o
valor de `alpha_escolhido` e rode de novo para ver o que muda na curva.

In [ ]:
alpha_escolhido = 0.01
w0_ex, w1_ex = 0.0, 0.0
historico_perda_ex = []

In [ ]:
# SEU CODIGO AQUI

In [ ]:
plt.plot(historico_perda_ex)
plt.xlabel("Passo")
plt.ylabel("MSE")
plt.title(f"Curva de perda com alpha = {alpha_escolhido}")
plt.show()
print(f"MSE depois de 100 passos: {historico_perda_ex[-1]:.2f}")

In [ ]:
if historico_perda_ex[-1] < historico_perda_ex[0]:
    print("✅ A perda caiu ao longo dos passos: o gradiente descendente está funcionando.")
else:
    print("❌ A perda não caiu. Confira se alpha_escolhido não está grande demais.")

### Exercício 5: treinando com o scikit-learn

Complete o código para treinar um `LinearRegression` chamado `meu_modelo`
com os dados de corridas. Ele resolve o mesmo problema por fórmula
fechada, sem passos.

In [ ]:
from sklearn.linear_model import LinearRegression

In [ ]:
# SEU CODIGO AQUI

In [ ]:
print(f"w0 encontrado: {meu_modelo.intercept_:.2f}")
print(f"w1 encontrado: {meu_modelo.coef_[0]:.2f}")

In [ ]:
if abs(meu_modelo.intercept_ - 4.6) < 1 and abs(meu_modelo.coef_[0] - 2.2) < 0.5:
    print("✅ Os coeficientes ficaram perto do esperado (taxa fixa ≈ 4,6; preço por km ≈ 2,2).")
else:
    print("❌ Confira se você usou dados[['distancia_km']] (com colchetes duplos) e dados['preco'].")

### Exercício 6: calculando as métricas

As quatro métricas desta aula:

$$\text{MAE} = \frac{1}{n}\sum_{i=1}^{n}\left|y_i - \hat{y}_i\right| \qquad \text{RMSE} = \sqrt{\text{MSE}}$$

$$\text{MAPE} = \frac{100}{n}\sum_{i=1}^{n}\left|\frac{y_i - \hat{y}_i}{y_i}\right| \qquad R^2 = 1 - \frac{\sum_{i=1}^{n}(y_i - \hat{y}_i)^2}{\sum_{i=1}^{n}(y_i - \bar{y})^2}$$

Use `mean_absolute_error`, `mean_squared_error` e `r2_score` do
`sklearn.metrics`. O MAPE não precisa de biblioteca: é o resíduo dividido
pelo preço real, em valor absoluto, e depois a média vezes 100.

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

precos_previstos_todos = meu_modelo.predict(dados[["distancia_km"]])
meus_residuos = dados["preco"] - precos_previstos_todos

In [ ]:
# SEU CODIGO AQUI

In [ ]:
print(f"MAE:  R$ {mae:.2f} (em média, o modelo erra esse valor por corrida)")
print(f"RMSE: R$ {rmse:.2f} (parecido com o MAE, mas penaliza mais os erros grandes)")
print(f"MAPE: {mape:.1f}% (o mesmo erro, como fração do preço de cada corrida)")
print(f"R²:   {r2:.2f} (a reta explica {r2 * 100:.0f}% da variação dos preços)")

In [ ]:
if r2 > 0.8 and 5 < mape < 20:
    print(f"✅ R² de {r2:.2f} e MAPE de {mape:.1f}%: os dois estão certos.")
    print("   Um diz que a reta explica quase tudo; o outro, que ela erra")
    print("   11% do preço numa corrida típica. Perguntas diferentes.")
else:
    print("❌ Confira o MAPE: divida o resíduo pelo preço real antes da média.")

### Exercício 7: desafio, uma previsão honesta

Preveja o preço de uma corrida de 12 km, e entregue a faixa em volta
dele em vez de um número solto.

São três passos. Calcule o resíduo típico $s$, monte a margem $2s$, e
aplique na previsão:

$$s = \sqrt{\frac{1}{n-2}\sum_{i=1}^{n}(y_i - \hat{y}_i)^2} \qquad \text{faixa} = \hat{y} \pm 2s$$

In [ ]:
corrida_nova = pd.DataFrame({"distancia_km": [12]})
n_corridas = len(dados)

In [ ]:
# SEU CODIGO AQUI

In [ ]:
print(f"Previsão para 12 km: R$ {preco_previsto_12km:.2f}")
print(f"Resíduo típico (s):  R$ {meu_s:.2f}")
print(f"Faixa: de R$ {preco_previsto_12km - minha_margem:.2f} "
      f"a R$ {preco_previsto_12km + minha_margem:.2f}")

In [ ]:
# A faixa funciona? Conte quantas das 200 corridas caem dentro da sua
dentro_da_faixa = meus_residuos.abs() <= minha_margem
print(f"{dentro_da_faixa.sum()} das {n_corridas} corridas caem na faixa "
      f"({100 * dentro_da_faixa.mean():.1f}%)")

In [ ]:
if 15 < preco_previsto_12km < 45 and 0.92 < dentro_da_faixa.mean() < 0.98:
    print("✅ A faixa cobre perto de 95% das corridas, como prometido.")
    print(f"   Essa é a diferença entre 'custa R$ {preco_previsto_12km:.2f}' e")
    print(f"   'custa entre R$ {preco_previsto_12km - minha_margem:.2f} e "
          f"R$ {preco_previsto_12km + minha_margem:.2f}'.")
    print("   Só a segunda frase é verdadeira.")
else:
    print("❌ Confira: s usa n - 2 no denominador, e a margem é 2 vezes s.")

Agora, em texto: um cliente pergunta quanto vai custar a corrida dele de
12 km. Escreva 2 a 3 frases respondendo, usando a faixa em vez do número
solto, e explique por que a faixa é a resposta mais honesta. Edite esta
célula (duplo clique nela) e escreva sua resposta no lugar deste
parágrafo.